In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks')


In [ ]:
!pip install torch --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 70.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 42.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 72.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 62.3 MB/s eta 0:00:00


In [ ]:
!pip install transformers accelerate bitsandbytes sentence-transformers einops
!pip install beautifulsoup4 pdfplumber lxml

  Using cached bitsandbytes-0.45.2-py3-none-manylinux_2_24_x86_64.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 9.8 MB/s eta 0:00:00


In [ ]:
import torch # Deep learnnig framework for GPU-accelerated tensor operations
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig # Hugging face tools for laoding and configuring language models
from sentence_transformers import SentenceTransformer # For generating text embeddings/vectors
import faiss # Fast similarity search and clustering of dense vectors
import numpy as np
from scrape import PaperScraper # For retrieving research papers

print(f"GPU available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")


GPU available: True
Number of GPUs: 1


In [ ]:
# Initialize the embedding model used for generating text embeddings
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# embedding_model = SentenceTransformer() # Load the BGE large model for high quality embeddings

# configure some quantization for efficient GPU usage
bnb_config = BitsAndBytesConfig(  # Configure 4-bit quantization settings
    load_in_4bit=True,  # Enable 4-bit quantization for reduced memory usage
    bnb_4bit_quant_type="nf4",  # Use normalized float4 quantization for better accuracy
    bnb_4bit_compute_dtype=torch.float16,  # Use float16 for compute to balance speed and precision
    bnb_4bit_use_double_quant=True  # Enable double quantization for additional memory savings
)

model_name = "zanchat/falcon-1b"
tokenizer = AutoTokenizer.from_pretrained(model_name) # Loads tokenizer fro converting text into tokens
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, # Apply the quantization settings
    torch_dtype=torch.float16, # Use float16 for model weights
    device_map='auto', # Automatically distribute model across available GPUs
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

The repository for zanchat/falcon-1b contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/zanchat/falcon-1b.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
The repository for zanchat/falcon-1b contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/zanchat/falcon-1b.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


model.safetensors:  75%|#######5  | 1.97G/2.62G [00:00<?, ?B/s]

In [ ]:
# Define a class that combines paper retrieval and embedding and question answer capabilities
class ResearchAssistant:
    def __init__(self):
        # Initialize components for paper processing and analysis
        self.scraper = PaperScraper()  # For retrieving papers from PubMed
        self.embedding_model = embedding_model  # For generating text embeddings
        self.tokenizer = tokenizer  # For tokenizing text for the LLM
        self.model = model  # The language model for answering questions
        self.paper_texts = []  # Store the full text of processed papers
        self.paper_metadata = []  # Store metadata (title, authors, etc) for papers
        self.index = None  # Will hold the FAISS similarity search index

    def search_papers(self, query: str, max_results: int = 10):
        """Search and download papers for the given query"""
        print(f"Searching for papers about: {query}")

        # Get paper IDs from PubMed search
        pmids = self.scraper.search_pubmed(query, max_results)
        # Fetch detailed information for each paper
        papers = self.scraper.fetch_pubmed_details(pmids)

        # Process each paper found
        for paper in papers:
            if pdf_url := paper.get('full_text_link'):  # Check if full text PDF is available
                try:
                    # Download PDF to temporary file
                    pdf_path = self.scraper.download_pdf(
                        pdf_url,
                        f"temp_{paper['pubmed_id']}.pdf"
                    )
                    # Extract plain text from PDF
                    text = self.scraper.extract_text_from_pdf(pdf_path)

                    # Save paper content and metadata
                    self.paper_texts.append(text)
                    self.paper_metadata.append(paper)

                except Exception as e:
                    print(f"Error processing paper {paper['pubmed_id']}: {e}")

        self._build_index()  # Create search index from processed papers
        print(f"Successfully processed {len(self.paper_texts)} papers")

    def _build_index(self):
        """Create FAISS index from paper embeddings"""
        # Initialize lists for storing chunks and their metadata
        self.chunks = []
        self.chunk_metadata = []

        # Process each paper into chunks
        for text, metadata in zip(self.paper_texts, self.paper_metadata):
            # Split text into paragraphs
            paragraphs = text.split('\n\n')
            # Create overlapping chunks of 3 paragraphs
            for i in range(0, len(paragraphs), 3):
                chunk = ' '.join(paragraphs[i:i+3])
                if len(chunk.split()) > 20:  # Only keep chunks with sufficient content
                    self.chunks.append(chunk)
                    self.chunk_metadata.append(metadata)

        # Generate embeddings for all chunks
        embeddings = self.embedding_model.encode(
            self.chunks,
            batch_size=4,  # Process 32 chunks at a time
            show_progress_bar=True,
            convert_to_numpy=True  # Convert to numpy for FAISS compatibility
        )

        # Initialize and populate FAISS index
        dimension = embeddings.shape[1]  # Get embedding dimension
        self.index = faiss.IndexFlatL2(dimension)  # Create L2 distance index
        self.index.add(embeddings)  # Add embeddings to index

    def answer_question(self, question: str, k: int = 5):
        """Answer a question using RAG"""
        # Convert question to embedding vector
        q_embedding = self.embedding_model.encode([question])[0]

        # Find k most similar chunks
        distances, indices = self.index.search(q_embedding.reshape(1, -1), k)

        # Build context from relevant chunks
        context = ""
        used_papers = set()  # Track which papers we've used
        for idx in indices[0]:
            chunk = self.chunks[idx]
            metadata = self.chunk_metadata[idx]
            paper_id = metadata['pubmed_id']

            # Only include first chunk from each paper
            if paper_id not in used_papers:
                used_papers.add(paper_id)
                context += f"\nFrom paper '{metadata['title']}':\n{chunk}\n"

        # Construct prompt for the LLM
        prompt = f"""Below is a scientific question and relevant excerpts from research papers.
        Please answer the question based on these excerpts. Include citations to the paper titles.

        Relevant excerpts:
        {context}

        Question: {question}

        Answer: """

        # Generate answer using LLM
        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")  # Tokenize and move to GPU
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=256,  # Limit response length
            temperature=0.7,  # Add some randomness to generation
            num_return_sequences=1,  # Generate one response
            do_sample=True,  # Use sampling instead of greedy decoding
        )

        # Extract and clean up the generated answer
        answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return answer.split("Answer: ")[-1].strip()











In [ ]:
# Initialize the assistant
assistant = ResearchAssistant()

# Search for papers on a topic
assistant.search_papers(
    query="latest developments in CRISPR gene editing cancer therapy",
    max_results=1
)

# Ask questions
questions = [
    "What are the main challenges in using CRISPR for cancer therapy?",
]

for question in questions:
    print(f"\nQ: {question}")
    print(f"\nA: {assistant.answer_question(question, k=1)}")

Searching for papers about: latest developments in CRISPR gene editing cancer therapy


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Successfully processed 1 papers

Q: What are the main challenges in using CRISPR for cancer therapy?


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.87 GiB. GPU 0 has a total capacity of 14.74 GiB of which 2.55 GiB is free. Process 290831 has 12.18 GiB memory in use. Of the allocated memory 9.56 GiB is allocated by PyTorch, and 2.50 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)